In [0]:
print("Spark version:", spark.version)
print("Databricks is ready!")


Spark version: 4.2.0
Databricks is ready!


In [0]:
data = [("Vancouver", "H001", 5.5, 120),
        ("Toronto", "H002", 8.2, 85),
        ("Calgary", "H003", 3.1, 200)]

columns = ["city", "hospital_id", "wait_time_hours", "patients_waiting"]

df = spark.createDataFrame(data, columns)
df.show()

+---------+-----------+---------------+----------------+
|     city|hospital_id|wait_time_hours|patients_waiting|
+---------+-----------+---------------+----------------+
|Vancouver|       H001|            5.5|             120|
|  Toronto|       H002|            8.2|              85|
|  Calgary|       H003|            3.1|             200|
+---------+-----------+---------------+----------------+



In [0]:
%pip install azure-eventhub

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Production Bronze Ingestion
# Canadian Healthcare Analytics Platform
# Author: Sai Krishna Reddy Kaithi

import json
import logging
from pyspark.sql.functions import (
    col, current_timestamp, lit, 
    from_json, schema_of_json
)
from pyspark.sql.types import (
    StructType, StructField, 
    StringType, DoubleType, 
    LongType, TimestampType
)

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Define schema for incoming health data
health_schema = StructType([
    StructField("city", StringType(), True),
    StructField("hospital_id", StringType(), True),
    StructField("wait_time_hours", DoubleType(), True),
    StructField("patients_waiting", LongType(), True),
    StructField("timestamp", DoubleType(), True),
    StructField("source", StringType(), True)
])

logger.info("Schema defined successfully")
print("Schema ready — production Bronze ingestion setup complete!")
print(health_schema)

INFO:__main__:Schema defined successfully


Schema ready — production Bronze ingestion setup complete!
StructType([StructField('city', StringType(), True), StructField('hospital_id', StringType(), True), StructField('wait_time_hours', DoubleType(), True), StructField('patients_waiting', LongType(), True), StructField('timestamp', DoubleType(), True), StructField('source', StringType(), True)])


In [0]:
pip install databricks-cli

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text(
    "eventhub_connection_string",
    "",
    "Event Hub Connection String"
)

CONNECTION_STRING = dbutils.widgets.get("eventhub_connection_string")
print("Widget created — paste connection string in box at top!")

Widget created — paste connection string in box at top!


In [0]:
# Paste your connection string here temporarily
CONNECTION_STRING = "Endpoint=sb://healthcare-analytics-eh.servicebus.windows.net/;SharedAccessKeyName=RootManageSharedAccessKey;SharedAccessKey=gywMC84AKpYWzkpDqxK4aW6bFxEgOIhus+AEhMkMQpw="
EVENTHUB_NAME = "health-data-stream"
print("Connection ready!")

Connection ready!


In [0]:
CONNECTION_STRING = "Endpoint=sb://healthcare-analytics-eh.servicebus.windows.net/;SharedAccessKeyName=RootManageSharedAccessKey;SharedAccessKey=gywMC84AKpYWzkpDqxK4aW6bFxEgOIhus+AEhMkMQpw="
EVENTHUB_NAME = "health-data-stream"
print("Connection ready!")

Connection ready!


In [0]:
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType
import random
import time

# Simulate healthcare data — same structure as Event Hubs
health_schema = StructType([
    StructField("city", StringType(), True),
    StructField("hospital_id", StringType(), True),
    StructField("wait_time_hours", DoubleType(), True),
    StructField("patients_waiting", LongType(), True),
    StructField("timestamp", DoubleType(), True),
    StructField("source", StringType(), True)
])

# Generate sample health records
cities = ["Vancouver", "Toronto", "Calgary", "Winnipeg", "Halifax"]
records = []
for i in range(100):
    records.append((
        random.choice(cities),
        f"H{random.randint(1,50):03d}",
        round(random.uniform(1, 24), 2),
        random.randint(10, 200),
        time.time(),
        "healthcare_simulation"
    ))

# Create DataFrame
df_bronze_raw = spark.createDataFrame(records, health_schema)

# Add production metadata
df_bronze = (df_bronze_raw
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_system", lit("event_hubs_simulation"))
    .withColumn("processing_date", current_timestamp().cast("date"))
)

print(f"✅ Generated {df_bronze.count()} health records")
df_bronze.show(5)

✅ Generated 100 health records
+---------+-----------+---------------+----------------+--------------------+--------------------+--------------------+--------------------+---------------+
|     city|hospital_id|wait_time_hours|patients_waiting|           timestamp|              source| ingestion_timestamp|       source_system|processing_date|
+---------+-----------+---------------+----------------+--------------------+--------------------+--------------------+--------------------+---------------+
|Vancouver|       H013|          18.61|             142| 1.789458822248223E9|healthcare_simula...|2026-09-15 07:54:...|event_hubs_simula...|     2026-09-15|
|  Toronto|       H017|           5.17|             116| 1.789458822248246E9|healthcare_simula...|2026-09-15 07:54:...|event_hubs_simula...|     2026-09-15|
|  Halifax|       H038|           1.17|             108|1.7894588222482617E9|healthcare_simula...|2026-09-15 07:54:...|event_hubs_simula...|     2026-09-15|
|  Calgary|       H036|    

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS healthcare_bronze")
print("✅ Schema created!")

✅ Schema created!


In [0]:
# Write to Delta Lake Bronze layer
(df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("healthcare_bronze.wait_times_raw")
)

print("✅ Bronze layer written successfully!")
print(f"Total records: {spark.table('healthcare_bronze.wait_times_raw').count()}")

✅ Bronze layer written successfully!
Total records: 100


In [0]:
df_verify = spark.table("healthcare_bronze.wait_times_raw")
print(f"✅ Total records in Bronze: {df_verify.count()}")
df_verify.show(5)

✅ Total records in Bronze: 100
+---------+-----------+---------------+----------------+--------------------+--------------------+--------------------+--------------------+---------------+
|     city|hospital_id|wait_time_hours|patients_waiting|           timestamp|              source| ingestion_timestamp|       source_system|processing_date|
+---------+-----------+---------------+----------------+--------------------+--------------------+--------------------+--------------------+---------------+
|Vancouver|       H013|          18.61|             142| 1.789458822248223E9|healthcare_simula...|2026-09-15 07:56:...|event_hubs_simula...|     2026-09-15|
|  Toronto|       H017|           5.17|             116| 1.789458822248246E9|healthcare_simula...|2026-09-15 07:56:...|event_hubs_simula...|     2026-09-15|
|  Halifax|       H038|           1.17|             108|1.7894588222482617E9|healthcare_simula...|2026-09-15 07:56:...|event_hubs_simula...|     2026-09-15|
|  Calgary|       H036|    